# Lumina LOB — Agents + Market Impact Demo

This notebook runs a minimal agent-based simulation and visualizes:

1. Reference price vs. best bid/ask quotes
2. Trade volume and signed volume per step
3. Market-maker inventory dynamics
4. Propagator-style temporary + permanent price impact

Everything is deterministic: fix the seeds and re-run to get the same path.

## Setup

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
from lumina_lob.agents import InformedTrader, MarketMaker, NoiseTrader
from lumina_lob.market_model import PropagatorImpact, ReferencePriceProcess
from lumina_lob.simulation import Simulation

## Build the market

- **Reference price**: low-volatility Brownian motion anchored near 100.
- **Noise trader**: random limit-order arrivals around the reference price.
- **Market maker**: symmetric quotes, tracks inventory via `on_fill`.
- **Informed trader**: bullish signal, trades with 30% probability each step.

In [ ]:
seed = 42
n_steps = 200

reference_price = ReferencePriceProcess(
    initial_price=100.0,
    drift=0.0,
    volatility=0.02,
    dt=1.0,
    seed=seed,
)

noise = NoiseTrader(
    seed=seed,
    arrival_rate=3,
    tick_size=1.0,
    side_bias=0.5,
)

market_maker = MarketMaker(
    spread_half_width=2,
    quote_size=10,
    tick_size=1.0,
    max_inventory=100,
)

informed = InformedTrader(
    signal="bullish",
    trade_size=20,
    participation_rate=0.3,
    order_type="market",
    seed=seed,
)

sim = Simulation(
    reference_price=reference_price,
    agents=[market_maker, noise, informed],
)

## Run the simulation

Step manually so we can also record the market maker's inventory each tick.

In [ ]:
inventory_history = []
for _ in range(n_steps):
    sim.step()
    inventory_history.append(market_maker.inventory)

df = sim.to_dataframe()
df["inventory"] = inventory_history
df.head(10)

## 1. Reference price vs. LOB quotes

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df["step"], df["reference_price"], label="reference price", alpha=0.7)
ax.plot(df["step"], df["best_bid"], label="best bid", drawstyle="steps-post")
ax.plot(df["step"], df["best_ask"], label="best ask", drawstyle="steps-post")
ax.set_xlabel("step")
ax.set_ylabel("price")
ax.set_title("Reference price and LOB quotes")
ax.legend()
plt.tight_layout()

## 2. Trade volume and signed volume

Signed volume is positive for buyer-initiated trades and negative for seller-initiated trades.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].bar(df["step"], df["trade_volume"], color="gray")
axes[0].set_ylabel("trade volume")
axes[0].set_title("Activity per step")
colors = ["green" if v >= 0 else "red" for v in df["signed_volume"]]
axes[1].bar(df["step"], df["signed_volume"], color=colors)
axes[1].set_ylabel("signed volume")
axes[1].set_xlabel("step")
plt.tight_layout()

## 3. Market-maker inventory dynamics

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df["step"], df["inventory"], label="market maker inventory")
ax.axhline(0, color="black", linestyle="--", linewidth=0.8)
ax.axhline(market_maker.max_inventory, color="red", linestyle="--", linewidth=0.8, label="inventory limit")
ax.axhline(-market_maker.max_inventory, color="red", linestyle="--", linewidth=0.8)
ax.set_xlabel("step")
ax.set_ylabel("inventory (shares)")
ax.set_title("Market-maker inventory over time")
ax.legend()
plt.tight_layout()

## 4. Propagator market impact

Feed the signed-volume series into a propagator impact model and compare the impacted price to the latent reference price.

In [ ]:
impact = PropagatorImpact(
    permanent_impact=0.001,
    temporary_impact=0.005,
    decay=0.9,
)

impacted_price = [reference_price.initial_price]
for signed_volume in df["signed_volume"]:
    impacted_price.append(impact.apply(signed_volume, impacted_price[-1]))

df["impacted_price"] = impacted_price[1:]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df["step"], df["reference_price"], label="reference price")
ax.plot(df["step"], df["impacted_price"], label="impacted price", linestyle="--")
ax.set_xlabel("step")
ax.set_ylabel("price")
ax.set_title("Propagator impact on the reference price")
ax.legend()
plt.tight_layout()

## Summary statistics

In [ ]:
summary = {
    "total_steps": len(df),
    "total_trade_volume": int(df["trade_volume"].sum()),
    "net_signed_volume": int(df["signed_volume"].sum()),
    "avg_spread": df["spread"].mean(),
    "final_inventory": int(market_maker.inventory),
    "impact_drift": float(df["impacted_price"].iloc[-1] - df["reference_price"].iloc[-1]),
}
pd.Series(summary).to_frame().T